# 한국의 환율을 이용해 역사를 알아보자

`데이터.db` 내의 뷰(`환율_정리표`, `환율_대통령_통합표`)를 활용하여 각 날짜별 환율 및 역대 정부별 환율 추이를 시각화합니다.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 한글 폰트 설정 (리눅스: NanumGothic, 윈도우: Malgun Gothic, 맥: AppleGothic)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. [환율_정리표 뷰] 각 날짜별 환율 추이 시각화
- `환율_정리표` 뷰는 날짜(연도, 월)와 환율이 숫자형으로 정제되어 있습니다.

In [ ]:
# 1. 데이터베이스 뷰에서 데이터 조회
conn = sqlite3.connect('데이터.db')
query = 'SELECT * FROM 환율_정리표 ORDER BY 연도, 월'
df_rate = pd.read_sql(query, conn)
conn.close()

# 2. 날짜 컬럼 생성 (연도 + 월 -> Datetime)
df_rate['날짜'] = pd.to_datetime(df_rate['연도'].astype(str) + '-' + df_rate['월'].astype(str) + '-01')

# 3. 시각화
plt.figure(figsize=(14, 6), dpi=120)
plt.plot(df_rate['날짜'], df_rate['환율'], color='#1f77b4', linewidth=2, label='원/달러 환율')

# 최고 환율 지점 강조 (1997~1998 IMF 외환위기)
max_row = df_rate.loc[df_rate['환율'].idxmax()]
plt.scatter([max_row['날짜']], [max_row['환율']], color='crimson', s=80, zorder=5)
plt.annotate(
    f"최고 환율: {max_row['환율']:,.1f}원\n({max_row['연도']}년 {max_row['월']}월)",
    xy=(max_row['날짜'], max_row['환율']),
    xytext=(max_row['날짜'] - pd.DateOffset(years=4), max_row['환율'] - 80),
    arrowprops=dict(facecolor='crimson', shrink=0.08, width=1.5, headwidth=7),
    fontsize=10, fontweight='bold', color='crimson'
)

plt.title('[환율_정리표 뷰 기준] 날짜별 원/달러 환율 추이 (1964 ~ 2002)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('날짜 (연도)', fontsize=12)
plt.ylabel('환율 (원/달러)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

# X축 눈금 간격 설정 (5년 단위)
plt.gca().xaxis.set_major_locator(mdates.YearLocator(5))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

## 2. [환율_대통령_통합표 뷰] 역대 정부(대통령)별 환율 변동 추이 시각화
- `환율_대통령_통합표` 뷰를 활용해 각 대통령 재임 시기별 환율 변화를 배경색과 라벨로 함께 확인합니다.

In [ ]:
# 1. 데이터베이스 뷰에서 데이터 조회
conn = sqlite3.connect('데이터.db')
query = '''
SELECT DISTINCT 연도, 월, 대통령, 환율 
FROM 환율_대통령_통합표 
ORDER BY 연도, 월
'''
df_pres = pd.read_sql(query, conn)
conn.close()

# 정권 교체 월 중복 처리 및 날짜 변환
df_pres = df_pres.drop_duplicates(subset=['연도', '월'], keep='last').copy()
df_pres['날짜'] = pd.to_datetime(df_pres['연도'].astype(str) + '-' + df_pres['월'].astype(str) + '-01')
df_pres = df_pres.sort_values('날짜').reset_index(drop=True)

# 2. 시각화
fig, ax = plt.subplots(figsize=(15, 7), dpi=120)
ax.plot(df_pres['날짜'], df_pres['환율'], color='black', linewidth=1.8, label='원/달러 환율', zorder=4)

# 대통령별 색상 매핑
presidents = df_pres['대통령'].unique()
colors = ['#E3F2FD', '#FFF3E0', '#E8F5E9', '#F3E5F5', '#FBE9E7', '#E0F7FA']
color_map = {p: colors[i % len(colors)] for i, p in enumerate(presidents)}

# 대통령 재임 구간별 배경 음영 및 라벨 표시
changes = df_pres[df_pres['대통령'] != df_pres['대통령'].shift(1)]
for i in range(len(changes)):
    start_date = changes.iloc[i]['날짜']
    pres_name = changes.iloc[i]['대통령']
    end_date = changes.iloc[i+1]['날짜'] if i+1 < len(changes) else df_pres['날짜'].max()
    
    ax.axvspan(start_date, end_date, color=color_map[pres_name], alpha=0.6, zorder=1)
    mid_date = start_date + (end_date - start_date) / 2
    ax.text(mid_date, df_pres['환율'].max() * 0.92, pres_name, 
            horizontalalignment='center', fontsize=11, fontweight='bold', color='#333333', zorder=5)

ax.set_title('[환율_대통령_통합표 뷰 기준] 역대 대통령별 원/달러 환율 변동 추이', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('날짜 (연도)', fontsize=12)
ax.set_ylabel('환율 (원/달러)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5, zorder=2)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()